In [1]:
# 0) SETUP: imports, download/leitura dos dados, preparação e split

import sys
import subprocess
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

# Tentar ler direto da URL; se preferir, use !wget no shell do notebook
URL = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
df = pd.read_csv(URL)

# 1) Preenchimento de valores ausentes com zero
df = df.fillna(0)

# 2) Definir target e features
target_col = 'fuel_efficiency_mpg'
y = df[target_col].values
df_features = df.drop(columns=[target_col])

# 3) Split 60/20/20 (com random_state=1)
df_full_train, df_test, y_full_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=1
)
df_train, df_val, y_train, y_val = train_test_split(
    df_full_train, y_full_train, test_size=0.25, random_state=1
)  # 0.25 de 0.8 => 0.2

# 4) DictVectorizer (sparse=True)
dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient='records')
val_dicts   = df_val.to_dict(orient='records')
test_dicts  = df_test.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val   = dv.transform(val_dicts)
X_test  = dv.transform(test_dicts)

feature_names = dv.get_feature_names_out()

print("Tamanhos:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Ex. de features:", feature_names[:10])


Tamanhos:
Train: (5822, 14) Val: (1941, 14) Test: (1941, 14)
Ex. de features: ['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'engine_displacement' 'fuel_type=Diesel'
 'fuel_type=Gasoline' 'horsepower' 'model_year' 'num_cylinders'
 'num_doors']


In [2]:
# Q1: DecisionTreeRegressor com max_depth=1 e feature do primeiro split

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Índice da feature usada na raiz (nó 0):
root_feature_idx = dt.tree_.feature[0]
split_feature = feature_names[root_feature_idx]

# Para comparar com as alternativas, vamos pegar o "nome base"
# (para variáveis categóricas one-hot, remove o sufixo "=valor")
split_base = split_feature.split('=')[0]

print("Feature de split (nome completo):", split_feature)
print("Feature base:", split_base)

# Opções do enunciado
opcoes = ['vehicle_weight', 'model_year', 'origin', 'fuel_type']

def escolhe_alternativa(nome_base):
    # Para 'origin' e 'fuel_type', se qualquer dummy gerou o split, respondemos pelo nome base
    # Para numéricas, o nome já coincide
    # Se não bater exatamente, escolhemos a 'mais próxima' por regra simples
    if nome_base in opcoes:
        return nome_base
    # Heurística: se começa com "origin=" ou "fuel_type=", mapeia para base
    if split_feature.startswith('origin='):
        return 'origin'
    if split_feature.startswith('fuel_type='):
        return 'fuel_type'
    # Caso contrário, procura a mais "similar"
    from difflib import get_close_matches
    m = get_close_matches(nome_base, opcoes, n=1)
    return m[0] if m else nome_base

print("Resposta (alternativa):", escolhe_alternativa(split_base))


Feature de split (nome completo): vehicle_weight
Feature base: vehicle_weight
Resposta (alternativa): vehicle_weight
